In [1]:
# %%
"""
GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
=============================================================
GPU-native PTQ benchmarking three GPU-capable approaches:

  1. ONNX Runtime  — FP32 → ONNX → INT8 (QDQ) via ORT quantization API
                     3 calibration methods: MinMax, Entropy, Percentile
                     Inference via CUDAExecutionProvider
  2. TensorRT      — FP32 → ONNX → TRT INT8 engine with calibration
                     IInt8EntropyCalibrator2; layer fusion (Conv+BN+ReLU)
                     (requires tensorrt + pycuda)
  3. PyTorch CUDA  — (a) Dynamic PTQ: Linear → qint8 on CUDA tensors
                     (b) PT2E: torch.export + X86InductorQuantizer → torch.compile

Mathematical foundation:
  Q(x)  = clip(round(x / S) + Z, q_min, q_max)   [quantization]
  x̂     = S · (Q(x) − Z)                          [dequantization]
  ε     = x − x̂,  |ε| ≤ S/2                      [bounded error]
  S     = (x_max − x_min) / (2^b − 1)             [scale from calibration]
  Z     = clip(round(−x_min / S), q_min, q_max)   [zero-point]

  Calibration method comparison:
  MinMax     : S = (x_max − x_min) / 255  — widest range, sensitive to outliers
  Entropy    : threshold T* = argmin KL(P_orig || P_quant) — robust to outliers
  Percentile : clips activation range at fixed percentile before computing S/Z

FLOPs note:
  FLOPs (op count) are IDENTICAL for FP32 and INT8.
  Quantization changes the storage dtype, NOT the number of operations.
  GPU speedup is a hardware property: INT8 Tensor Core lanes process
  ~4× more elements per cycle than FP32 (Ampere), ~8× (Hopper).

FIXES in this version vs original:
  Fix 1  — Calibration reader exhausted bug: each ORT method now gets its
            own fresh DataLoader (get_calib_loader() called per method),
            so MinMax/Entropy/Percentile each see the full calibration set.
            Original shared one iterator → entropy/percentile got 0 batches.
  Fix 2  — ORT provider verification: active provider logged after session
            creation; silent CPU fallback now surfaced in output JSON.
  Fix 3  — ORT timing fairness: wall-clock warmup raised from 5 → 20 to
            match GPU timing; timing_method field clearly states wall-clock
            so report readers know it is not directly comparable to CUDA events.
  Fix 4  — count_params: canonical two-path deduplication (id(p) for
            nn.Parameter, module-path key for FX callable weights).
            Original used bare sum(p.numel()) — no dedup, misses FX weights.
  Fix 5  — disk_size_mb for PyTorch models: uses torch.save(model) not
            state_dict() for consistent serialization across all methods.
            ONNX/TRT files measured by file path (correct, unchanged).
  Fix 6  — GPU warmup raised from 10 → 20 for both batch and single-image.
  Fix 7  — TRT/ORT warmup raised from 5 → 20.
  Fix 8  — _best_result section added to JSON output.
  Fix 9  — dynamic_ptq disk size measured consistently via torch.save(model).
  Fix 10 — ORT size note added: QDQ format adds graph metadata overhead so
            INT8 ONNX files can be larger than FP32 torch.save; this is a
            format artifact, not a real model size increase.
  Fix 11 — benchmark_fp32 uses canonical count_params.
  Fix 12 — _meta section added to JSON with environment and method notes.

Output:
  __1__PTQ_GPU.json
  __1__saved_models_PTQ_GPU/
"""

# ── Imports ────────────────────────────────────────────────────────────────────
import os
import sys
import json
import time
import copy
import shutil
import tempfile
import warnings
import argparse
import subprocess
from typing import Dict, List, Optional

import numpy as np

warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__1__PTQ_GPU.json"
SAVE_DIR              = "__1__saved_models_PTQ_GPU"

BATCH_SIZE     = 128
NUM_WORKERS    = 4
NUM_CLASSES    = 10
CALIB_SIZE     = 1024
CALIB_BATCHES  = 8
INFERENCE_RUNS = 50
WARMUP_RUNS    = 20          # Fix 6: raised from 10 → 20, applied everywhere
INPUT_SHAPE    = (1, 3, 32, 32)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── Lazy dependency imports ────────────────────────────────────────────────────
def _try_import(name: str):
    try:
        import importlib
        return importlib.import_module(name)
    except ImportError:
        return None

torch       = _try_import("torch")
torchvision = _try_import("torchvision")
ort         = _try_import("onnxruntime")
onnx        = _try_import("onnx")
fvcore      = _try_import("fvcore")
tensorrt    = _try_import("tensorrt")
pycuda      = _try_import("pycuda")


def install_deps():
    cmds = [
        "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
        "pip install onnx onnxruntime-gpu",
        "pip install fvcore",
        "pip install tensorrt pycuda",
    ]
    for cmd in cmds:
        print(f"  $ {cmd}")
        subprocess.run(cmd.split(), check=False)


# ══════════════════════════════════════════════════════════════════════════════
# Model & Data
# ══════════════════════════════════════════════════════════════════════════════
def build_model(num_classes: int = NUM_CLASSES):
    import torch.nn as nn
    from torchvision import models
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model


def load_baseline(path: str):
    model = build_model(NUM_CLASSES)
    model.load_state_dict(
        torch.load(path, map_location="cpu", weights_only=True)
    )
    model.eval()
    return model


def get_test_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = datasets.CIFAR10(root="../data", train=False, download=True, transform=tf)
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )


def get_calib_loader():
    """
    Fix 1: returns a NEW DataLoader each call with a fresh iterator.
    Must be called separately for each calibration method — never share
    one loader across methods or the iterator will be exhausted.
    """
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, Subset
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = datasets.CIFAR10(root="../data", train=True, download=True, transform=tf)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(
        sub, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )


# ══════════════════════════════════════════════════════════════════════════════
# Parameter Counting  (Fix 4)
# Canonical two-path deduplication — handles FP32 and FX-quantized models.
# ══════════════════════════════════════════════════════════════════════════════
def count_params(model, model_label: str = "") -> Dict:
    """
    ① nn.Parameter objects  → deduplicate by id(p).
      id() is safe: each Parameter is a distinct Python object.

    ② FX-quantized .weight() callables → deduplicate by (module_name, 'weight').
      convert_fx() ops expose weights via .weight() returning a fresh
      dequantized FloatTensor on each call; id() and data_ptr() are unstable.

    Why NOT data_ptr():
      Views/slices share data_ptr() with their base tensor, causing
      silent undercounting when used as a dedup key.
    """
    import torch.nn as nn
    total, nonzero   = 0, 0
    seen_param_ids   = set()
    seen_module_keys = set()

    for mod_name, module in model.named_modules():
        # ① Standard parameters
        for _, p in module.named_parameters(recurse=False):
            pid = id(p)
            if pid in seen_param_ids:
                continue
            seen_param_ids.add(pid)
            total   += p.numel()
            nonzero += int((p != 0).sum().item())

        # ② FX quantized callable weights
        if hasattr(module, "weight") and callable(module.weight):
            key = (mod_name, "weight")
            if key in seen_module_keys:
                continue
            seen_module_keys.add(key)
            try:
                w = module.weight()
                if isinstance(w, torch.Tensor) and w.numel() > 0:
                    total   += w.numel()
                    nonzero += int((w != 0).sum().item())
            except Exception:
                pass

    if model_label:
        print(
            f"      count_params [{model_label}]: "
            f"total={total:,}  nonzero={nonzero:,}  "
            f"sparsity={1 - nonzero / max(total, 1):.4f}"
        )
    return {"params_total": int(total), "params_nonzero": int(nonzero)}


# ══════════════════════════════════════════════════════════════════════════════
# FLOPs
# fvcore preferred; manual hook fallback.
# Computed ONCE on FP32 and reused — op count is identical for INT8.
# ══════════════════════════════════════════════════════════════════════════════
def count_flops(model) -> Dict:
    """
    FLOPs = 2 × MACs
      Conv2d : MACs = (Cin/groups) × Cout × Kh × Kw × Hout × Wout
      Linear : MACs = in_features × out_features

    FLOPs are IDENTICAL for FP32 and INT8 — quantization changes dtype,
    not arithmetic operation count.
    """
    import torch.nn as nn

    if fvcore is not None:
        try:
            from fvcore.nn import FlopCountAnalysis
            x     = torch.randn(*INPUT_SHAPE)
            flops = FlopCountAnalysis(model.cpu().eval(), x)
            flops.unsupported_ops_warnings(False)
            total_macs   = flops.total()
            params_total = sum(p.numel() for p in model.parameters())
            return {
                "flops_G"     : round(total_macs * 2 / 1e9, 9),
                "params_total": int(params_total),
                "params_M"    : round(params_total / 1e6, 3),
                "tool"        : "fvcore",
            }
        except Exception:
            pass

    # Manual hook fallback
    spatial, handles = {}, []

    def make_hook(name):
        def hook(_, __, out):
            if hasattr(out, "shape"):
                spatial[name] = tuple(out.shape)
        return hook

    for name, mod in model.named_modules():
        handles.append(mod.register_forward_hook(make_hook(name)))
    x = torch.randn(*INPUT_SHAPE)
    with torch.no_grad():
        model.cpu().eval()(x)
    for h in handles:
        h.remove()

    total_flops = 0
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d):
            shape = spatial.get(name)
            if shape is None:
                continue
            _, cout, hout, wout = shape
            kh = mod.kernel_size[0] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            kw = mod.kernel_size[1] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            total_flops += 2 * cout * (mod.in_channels // mod.groups) * kh * kw * hout * wout
        elif isinstance(mod, nn.Linear):
            total_flops += 2 * mod.in_features * mod.out_features

    params_total = sum(p.numel() for p in model.parameters())
    return {
        "flops_G"     : round(total_flops / 1e9, 9),
        "params_total": int(params_total),
        "params_M"    : round(params_total / 1e6, 3),
        "tool"        : "manual",
    }


# ══════════════════════════════════════════════════════════════════════════════
# Disk Size  (Fix 5)
# PyTorch models: torch.save(model) — full model, consistent across FP32/INT8.
# ONNX/TRT files: measured by file path (no change needed).
# ══════════════════════════════════════════════════════════════════════════════
def disk_size_mb(obj) -> float:
    """
    Pass a file path (str) for ONNX/TRT files, or a PyTorch model object.
    PyTorch models are serialised via torch.save(model) — NOT state_dict() —
    so FP32 and quantized models are measured with the same format.

    Note for ONNX: QDQ format adds QuantizeLinear/DequantizeLinear node pairs
    on top of weights, so INT8 ONNX files can be LARGER than FP32 torch.save.
    This is a serialization format artifact, not a real model size increase.
    """
    if isinstance(obj, str):
        return os.path.getsize(obj) / 1024 ** 2
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(obj, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


# ══════════════════════════════════════════════════════════════════════════════
# Inference Timing
# ══════════════════════════════════════════════════════════════════════════════
def measure_gpu_throughput(model, device, n: int = INFERENCE_RUNS) -> Dict:
    """
    CUDA event timing — accurate GPU-only measurement, excludes Python overhead.
    Fix 6: warmup raised from 10 → 20 for both batch and single-image.
    """
    model    = model.to(device).eval()
    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)

    dummy_batch  = torch.randn(BATCH_SIZE, 3, 32, 32, device=device)
    dummy_single = torch.randn(1,          3, 32, 32, device=device)

    # Batch warmup
    with torch.no_grad():
        for _ in range(WARMUP_RUNS):
            model(dummy_batch)
    torch.cuda.synchronize(device)

    # Batch timing
    batch_timings = []
    with torch.no_grad():
        for _ in range(n):
            start_ev.record()
            model(dummy_batch)
            end_ev.record()
            torch.cuda.synchronize()
            batch_timings.append(start_ev.elapsed_time(end_ev))

    # Single warmup
    with torch.no_grad():
        for _ in range(WARMUP_RUNS):
            model(dummy_single)
    torch.cuda.synchronize(device)

    # Single timing
    single_timings = []
    with torch.no_grad():
        for _ in range(n):
            start_ev.record()
            model(dummy_single)
            end_ev.record()
            torch.cuda.synchronize()
            single_timings.append(start_ev.elapsed_time(end_ev))

    batch_ms  = float(np.mean(batch_timings))
    single_ms = float(np.mean(single_timings))
    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
        "warmup_runs"        : WARMUP_RUNS,
        "timing_runs"        : n,
    }


def measure_ort_throughput(session, n: int = INFERENCE_RUNS) -> Dict:
    """
    Wall-clock timing for ORT sessions — CUDA event API not available here.
    Fix 3: warmup raised from 5 → 20.
    Fix 3: timing_method field explicitly states wall-clock so downstream
           consumers know this is NOT directly comparable to CUDA event timings.
    """
    input_name   = session.get_inputs()[0].name
    dummy_batch  = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
    dummy_single = np.random.randn(1,          3, 32, 32).astype(np.float32)

    # Fix 7: warmup raised from 5 → 20
    for _ in range(WARMUP_RUNS):
        session.run(None, {input_name: dummy_batch})

    t0 = time.perf_counter()
    for _ in range(n):
        session.run(None, {input_name: dummy_batch})
    batch_ms = (time.perf_counter() - t0) / n * 1000

    for _ in range(WARMUP_RUNS):
        session.run(None, {input_name: dummy_single})
    t0 = time.perf_counter()
    for _ in range(n):
        session.run(None, {input_name: dummy_single})
    single_ms = (time.perf_counter() - t0) / n * 1000

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
        "timing_method"      : (
            "wall-clock (time.perf_counter) — ORT session; "
            "NOT directly comparable to CUDA event timings"
        ),
        "warmup_runs"        : WARMUP_RUNS,
        "timing_runs"        : n,
    }


# ══════════════════════════════════════════════════════════════════════════════
# Canonical Result Entry Builder
# ══════════════════════════════════════════════════════════════════════════════
def build_entry(metrics: Dict, params: Dict,
                size_mb: Optional[float], flops_G: float,
                inference_ms: Dict,
                extra_notes: Optional[Dict] = None) -> Dict:
    """
    Canonical output schema shared across all methods and scripts.
    size_mb may be None for compiled models that cannot be trivially serialised.
    extra_notes: optional dict of method-specific annotations added to entry.
    """
    entry = {
        "accuracy"      : round(metrics["accuracy"],  6),
        "precision"     : round(metrics["precision"], 6),
        "recall"        : round(metrics["recall"],    6),
        "f1"            : round(metrics["f1"],        6),
        "params"        : int(params["params_total"]),
        "params_nonzero": int(params["params_nonzero"]),
        "size_mb"       : round(size_mb, 6) if size_mb is not None else None,
        "flops_G"       : round(flops_G, 9),
        "inference_ms"  : inference_ms,
    }
    if extra_notes:
        entry.update(extra_notes)
    return entry


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate_torch(model, loader, device) -> Dict:
    from sklearn.metrics import (accuracy_score, precision_score,
                                 recall_score, f1_score)
    model = model.to(device).eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        preds.extend(model(inputs.to(device)).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def evaluate_onnx(session, loader) -> Dict:
    from sklearn.metrics import (accuracy_score, precision_score,
                                 recall_score, f1_score)
    input_name = session.get_inputs()[0].name
    preds, labels = [], []
    for inputs, lbls in loader:
        out = session.run(None, {input_name: inputs.numpy()})[0]
        preds.extend(np.argmax(out, axis=1))
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


# ══════════════════════════════════════════════════════════════════════════════
# ONNX Export
# ══════════════════════════════════════════════════════════════════════════════
def export_to_onnx(model, output_path: str, opset: int = 17) -> str:
    """
    opset 17+ required for QDQ INT8 ops.
    Dynamic axes allow variable batch size at inference.
    """
    dummy = torch.randn(*INPUT_SHAPE)
    torch.onnx.export(
        model.cpu().eval(), dummy, output_path,
        opset_version       = opset,
        input_names         = ["input"],
        output_names        = ["output"],
        dynamic_axes        = {"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        do_constant_folding = True,
        export_params       = True,
    )
    print(
        f"        ✓ ONNX exported → {output_path}  "
        f"({os.path.getsize(output_path) / 1024**2:.2f} MB, opset={opset})"
    )
    return output_path


# ══════════════════════════════════════════════════════════════════════════════
# 1. ONNX Runtime — INT8 Static PTQ  (3 calibration methods)
# ══════════════════════════════════════════════════════════════════════════════
def run_onnx_ptq(fp32_model, test_loader, flops_G: float,
                 baseline_params: Dict) -> Dict:
    """
    Static INT8 quantization via onnxruntime-gpu.

    QDQ format: QuantizeLinear/DequantizeLinear node pairs wrap each op.
    Weights:    INT8 per-channel symmetric
    Activations: UINT8 per-tensor affine

    Fix 1: get_calib_loader() called INSIDE the per-method loop so each
           method gets a fresh iterator. Original shared one iterator —
           entropy/percentile received 0 calibration batches.

    Fix 2: active ORT provider verified and logged after session creation.

    Fix 10: size note added — QDQ ONNX files include graph metadata overhead
            so can be larger than FP32 torch.save; documented in entry notes.
    """
    import onnxruntime
    from onnxruntime.quantization import (
        quantize_static, CalibrationDataReader,
        QuantType, QuantFormat, CalibrationMethod,
    )

    print("\n  [1/3] ONNX Runtime — Static INT8 PTQ (QDQ)")
    os.makedirs(SAVE_DIR, exist_ok=True)

    fp32_onnx = os.path.join(SAVE_DIR, "resnet50_fp32_tmp.onnx")
    export_to_onnx(fp32_model, fp32_onnx)

    class CIFARCalibReader(CalibrationDataReader):
        """
        Fix 1: constructed from a fresh loader each time.
        Never re-use across methods — the iterator is stateful and exhausted
        after the first calibration run.
        """
        def __init__(self, loader, max_batches: int = CALIB_BATCHES):
            self._iter    = iter(loader)
            self._batches = 0
            self._max     = max_batches

        def get_next(self):
            if self._batches >= self._max:
                return None
            try:
                imgs, _ = next(self._iter)
                self._batches += 1
                return {"input": np.ascontiguousarray(imgs.numpy(), dtype=np.float32)}
            except StopIteration:
                return None

    calib_cfg = {
        "ort_minmax"    : CalibrationMethod.MinMax,
        "ort_entropy"   : CalibrationMethod.Entropy,
        "ort_percentile": CalibrationMethod.Percentile,
    }

    available_providers = onnxruntime.get_available_providers()
    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if "CUDAExecutionProvider" in available_providers
        else ["CPUExecutionProvider"]
    )
    print(f"        ORT available providers: {available_providers}")
    print(f"        ORT requested providers: {providers}")

    sess_opts = onnxruntime.SessionOptions()
    sess_opts.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL

    results = {}

    for key, calib_method in calib_cfg.items():
        label     = key.replace("ort_", "").capitalize()
        int8_tmp  = os.path.join(SAVE_DIR, f"resnet50_int8_{label.lower()}_tmp.onnx")
        save_path = os.path.join(SAVE_DIR, f"onnx_int8_{label.lower()}.onnx")
        print(f"\n        [{label}] calibrating ...", flush=True)

        try:
            # Fix 1: fresh loader + fresh reader for every method
            fresh_calib_loader = get_calib_loader()

            quantize_static(
                model_input             = fp32_onnx,
                model_output            = int8_tmp,
                calibration_data_reader = CIFARCalibReader(fresh_calib_loader),
                quant_format            = QuantFormat.QDQ,
                per_channel             = True,
                reduce_range            = False,
                weight_type             = QuantType.QInt8,
                activation_type         = QuantType.QUInt8,
                calibrate_method        = calib_method,
                extra_options           = {
                    "EnableSubgraph"                  : True,
                    "MatMulConstBOnly"                : False,
                    "AddQDQPairToWeight"              : True,
                    "QuantizeAllFixedZeroPointTensors": True,
                },
            )

            session = onnxruntime.InferenceSession(
                int8_tmp, sess_opts, providers=providers
            )

            # Fix 2: verify and log which provider is actually running
            active_provider = session.get_providers()[0]
            print(f"        [{label}] active provider: {active_provider}")
            if active_provider == "CPUExecutionProvider" and "CUDAExecutionProvider" in providers:
                print(
                    f"        [{label}] ⚠ WARNING: fell back to CPU despite CUDA "
                    f"being available. Timing numbers are CPU, not GPU."
                )

            metrics  = evaluate_onnx(session, test_loader)
            size_mb  = disk_size_mb(int8_tmp)
            infer_ms = measure_ort_throughput(session)

            shutil.copy(int8_tmp, save_path)

            results[key] = build_entry(
                metrics      = metrics,
                params       = baseline_params,   # ONNX arch = FP32 arch
                size_mb      = size_mb,
                flops_G      = flops_G,
                inference_ms = infer_ms,
                extra_notes  = {
                    "active_provider": active_provider,
                    "calib_method"   : label,
                    "calib_batches"  : CALIB_BATCHES,
                    "calib_images"   : CALIB_SIZE,
                    # Fix 10: document the format size artifact
                    "size_note": (
                        "QDQ ONNX format adds QuantizeLinear/DequantizeLinear "
                        "node pairs to the graph; file can be larger than FP32 "
                        "torch.save — this is a serialization format artifact, "
                        "not a real model size increase."
                    ),
                },
            )

            print(
                f"        [{label}] Acc: {metrics['accuracy']:.4f}  "
                f"Disk: {size_mb:.2f} MB  "
                f"Batch: {infer_ms['batch128_gpu_ms']:.1f} ms  "
                f"Provider: {active_provider}"
            )
            print(f"        ✓ Saved → {save_path}")

        except Exception as exc:
            print(f"        [{label}] → FAILED: {exc}")
            results[key] = None
        finally:
            if os.path.exists(int8_tmp):
                os.remove(int8_tmp)

    # Keep FP32 ONNX for potential reuse by TRT; caller cleans up
    return results, fp32_onnx


# ══════════════════════════════════════════════════════════════════════════════
# 2. TensorRT — INT8 PTQ with IInt8EntropyCalibrator2
# ══════════════════════════════════════════════════════════════════════════════
def run_tensorrt_ptq(fp32_onnx: str, test_loader,
                     flops_G: float, baseline_params: Dict) -> Optional[Dict]:
    """
    TensorRT INT8 engine from ONNX via NVIDIA builder API.

    Calibration: IInt8EntropyCalibrator2
      S = T* / 127  where T* = argmin KL(P_orig || P_quant)
    Additional optimisations: Conv+BN+ReLU fusion, NHWC layout, kernel autotuning.
    Engine is GPU-architecture-specific — NOT portable across GPU generations.

    Fix 7: TRT calibrator warmup implicitly covered by CALIB_BATCHES.
    Fix 7: post-build inference warmup raised from 5 → WARMUP_RUNS (20).
    """
    print("\n  [2/3] TensorRT — INT8 PTQ")
    try:
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit  # noqa: F401
    except ImportError as exc:
        print(f"        ⚠ TensorRT/pycuda not available: {exc}")
        return None

    os.makedirs(SAVE_DIR, exist_ok=True)
    TRT_LOGGER  = trt.Logger(trt.Logger.WARNING)
    trt_tmp     = os.path.join(SAVE_DIR, "resnet50_int8_tmp.trt")
    calib_cache = os.path.join(SAVE_DIR, "trt_calib_tmp.cache")

    try:
        class CIFARCalibrator(trt.IInt8EntropyCalibrator2):
            def __init__(self, loader, cache_file: str):
                super().__init__()
                self._loader     = iter(loader)
                self._batches    = 0
                self._cache_file = cache_file
                first_batch, _   = next(iter(loader))
                self._batch_size = first_batch.shape[0]
                self._device_mem = cuda.mem_alloc(
                    int(np.prod(first_batch.shape) * 4)
                )

            def get_batch_size(self):
                return self._batch_size

            def get_batch(self, names):
                if self._batches >= CALIB_BATCHES:
                    return None
                try:
                    imgs, _ = next(self._loader)
                    cuda.memcpy_htod(
                        self._device_mem,
                        imgs.numpy().astype(np.float32),
                    )
                    self._batches += 1
                    return [int(self._device_mem)]
                except StopIteration:
                    return None

            def read_calibration_cache(self):
                if os.path.exists(self._cache_file):
                    with open(self._cache_file, "rb") as f:
                        return f.read()
                return None

            def write_calibration_cache(self, cache):
                with open(self._cache_file, "wb") as f:
                    f.write(cache)

        builder = trt.Builder(TRT_LOGGER)
        network = builder.create_network(
            1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        )
        parser = trt.OnnxParser(network, TRT_LOGGER)
        with open(fp32_onnx, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError(f"ONNX parse error: {parser.get_error(0)}")

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
        config.set_flag(trt.BuilderFlag.INT8)
        config.set_flag(trt.BuilderFlag.FP16)
        # Fix 1 (TRT): fresh calib loader
        config.int8_calibrator = CIFARCalibrator(get_calib_loader(), calib_cache)

        profile = builder.create_optimization_profile()
        profile.set_shape(
            "input",
            min=(1, 3, 32, 32),
            opt=(BATCH_SIZE, 3, 32, 32),
            max=(BATCH_SIZE * 2, 3, 32, 32),
        )
        config.add_optimization_profile(profile)

        print("        Building TRT INT8 engine (may take a few minutes) ...")
        t0         = time.perf_counter()
        serialized = builder.build_serialized_network(network, config)
        build_sec  = time.perf_counter() - t0
        print(f"        Engine built in {build_sec:.1f}s")

        with open(trt_tmp, "wb") as f:
            f.write(serialized)

        runtime = trt.Runtime(TRT_LOGGER)
        engine  = runtime.deserialize_cuda_engine(serialized)

        context = engine.create_execution_context()
        context.set_input_shape("input", (BATCH_SIZE, 3, 32, 32))
        d_input  = cuda.mem_alloc(BATCH_SIZE * 3 * 32 * 32 * 4)
        d_output = cuda.mem_alloc(BATCH_SIZE * NUM_CLASSES * 4)
        stream   = cuda.Stream()

        def infer_trt_batch(x_np: np.ndarray) -> np.ndarray:
            cuda.memcpy_htod_async(d_input, x_np.astype(np.float32), stream)
            context.execute_async_v2(
                bindings=[int(d_input), int(d_output)],
                stream_handle=stream.handle,
            )
            out = np.empty((BATCH_SIZE, NUM_CLASSES), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_output, stream)
            stream.synchronize()
            return out

        ctx1   = engine.create_execution_context()
        ctx1.set_input_shape("input", (1, 3, 32, 32))
        d_in1  = cuda.mem_alloc(1 * 3 * 32 * 32 * 4)
        d_out1 = cuda.mem_alloc(1 * NUM_CLASSES * 4)

        def infer_trt_single(x_np: np.ndarray) -> np.ndarray:
            cuda.memcpy_htod_async(d_in1, x_np.astype(np.float32), stream)
            ctx1.execute_async_v2(
                bindings=[int(d_in1), int(d_out1)],
                stream_handle=stream.handle,
            )
            out = np.empty((1, NUM_CLASSES), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_out1, stream)
            stream.synchronize()
            return out

        # Accuracy
        from sklearn.metrics import (accuracy_score, precision_score,
                                     recall_score, f1_score)
        preds, labels = [], []
        for inputs, lbls in test_loader:
            out = infer_trt_batch(inputs.numpy())
            preds.extend(np.argmax(out, axis=1))
            labels.extend(lbls.numpy())
        y_pred, y_true = np.array(preds), np.array(labels)
        metrics = {
            "accuracy" : float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        }

        # Timing — Fix 7: warmup raised from 5 → WARMUP_RUNS
        dummy_b = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
        dummy_s = np.random.randn(1,          3, 32, 32).astype(np.float32)
        start_ev = cuda.Event()
        end_ev   = cuda.Event()

        for _ in range(WARMUP_RUNS):
            infer_trt_batch(dummy_b)
            infer_trt_single(dummy_s)

        batch_times, single_times = [], []
        for _ in range(INFERENCE_RUNS):
            start_ev.record()
            infer_trt_batch(dummy_b)
            end_ev.record()
            end_ev.synchronize()
            batch_times.append(start_ev.time_till(end_ev))

            start_ev.record()
            infer_trt_single(dummy_s)
            end_ev.record()
            end_ev.synchronize()
            single_times.append(start_ev.time_till(end_ev))

        batch_ms  = float(np.mean(batch_times))
        single_ms = float(np.mean(single_times))
        infer_ms  = {
            "single_img_gpu_ms"  : round(single_ms, 4),
            "batch128_gpu_ms"    : round(batch_ms, 4),
            "per_img_gpu_ms"     : round(batch_ms / BATCH_SIZE, 4),
            "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
            "timing_method"      : "CUDA events (pycuda) + stream.synchronize()",
            "warmup_runs"        : WARMUP_RUNS,
            "timing_runs"        : INFERENCE_RUNS,
        }

        save_path = os.path.join(SAVE_DIR, "tensorrt_int8.trt")
        shutil.copy(trt_tmp, save_path)
        size_mb = disk_size_mb(trt_tmp)

        result = build_entry(
            metrics      = metrics,
            params       = baseline_params,
            size_mb      = size_mb,
            flops_G      = flops_G,
            inference_ms = infer_ms,
            extra_notes  = {
                "calibrator"    : "IInt8EntropyCalibrator2",
                "calib_batches" : CALIB_BATCHES,
                "calib_images"  : CALIB_SIZE,
                "build_time_s"  : round(build_sec, 2),
                "fp16_fallback" : True,
            },
        )

        print(
            f"        → Acc: {metrics['accuracy']:.4f}  "
            f"Disk: {size_mb:.2f} MB  "
            f"Batch: {batch_ms:.1f} ms"
        )
        print(f"        ✓ Saved → {save_path}")
        return result

    except Exception as exc:
        print(f"        → FAILED: {exc}")
        return None
    finally:
        for f in [trt_tmp, calib_cache]:
            if os.path.exists(f):
                os.remove(f)


# ══════════════════════════════════════════════════════════════════════════════
# 3a. PyTorch Dynamic PTQ on CUDA
# ══════════════════════════════════════════════════════════════════════════════
def run_dynamic_ptq(fp32_model, test_loader,
                    flops_G: float, device) -> Optional[Dict]:
    """
    Dynamic PTQ: only Linear weights stored as INT8.
    On CUDA, weights are dequantized to FP16/FP32 before cuBLAS GEMM —
    NOT true INT8 GEMM. Benefit: ~2× bandwidth reduction for Linear layers.
    Conv2d stays FP32. No calibration data required.

    Fix 9: disk_size_mb uses torch.save(model) not state_dict().
    """
    import torch.nn as nn
    print("\n        3a. Dynamic PTQ (Linear → qint8) on CUDA")
    os.makedirs(SAVE_DIR, exist_ok=True)

    try:
        q_dyn = torch.quantization.quantize_dynamic(
            copy.deepcopy(fp32_model), {nn.Linear}, dtype=torch.qint8
        )
        q_dyn.eval()

        # Fix 9: save full model for consistent size measurement
        save_path = os.path.join(SAVE_DIR, "dynamic_ptq_int8.pt")
        torch.save(q_dyn.cpu(), save_path)
        print(f"        ✓ Saved → {save_path}")

        q_dyn    = q_dyn.to(device)
        metrics  = evaluate_torch(q_dyn, test_loader, device)
        infer_ms = measure_gpu_throughput(q_dyn, device)
        params   = count_params(q_dyn.cpu(), model_label="dynamic_ptq")
        size_mb  = disk_size_mb(q_dyn.cpu())   # Fix 9: torch.save(model)

        result = build_entry(
            metrics      = metrics,
            params       = params,
            size_mb      = size_mb,
            flops_G      = flops_G,
            inference_ms = infer_ms,
            extra_notes  = {
                "note": (
                    "Dynamic PTQ quantizes Linear weights only. "
                    "On CUDA, weights are dequantized before cuBLAS GEMM — "
                    "NOT true INT8 GEMM. Conv2d stays FP32."
                )
            },
        )
        print(
            f"           → Acc: {metrics['accuracy']:.4f}  "
            f"GPU: {infer_ms['batch128_gpu_ms']:.1f} ms  "
            f"Disk: {size_mb:.2f} MB"
        )
        return result

    except Exception as exc:
        print(f"           → FAILED: {exc}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# 3b. PT2E + X86InductorQuantizer → torch.compile
# ══════════════════════════════════════════════════════════════════════════════
def _resolve_pt2e_imports():
    errors = []
    prepare_pt2e = convert_pt2e = None
    for mod_path in [
        "torch.ao.quantization.quantize_pt2e",
        "torch.quantization.quantize_pt2e",
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            prepare_pt2e = m.prepare_pt2e
            convert_pt2e = m.convert_pt2e
            break
        except (ImportError, AttributeError) as exc:
            errors.append(f"  {mod_path}: {exc}")

    if prepare_pt2e is None:
        raise ImportError(
            "Cannot find prepare_pt2e / convert_pt2e.\n"
            + "\n".join(errors) + "\nRequires PyTorch >= 2.3."
        )

    Quantizer = get_config = None
    for mod_path, cls, cfg in [(
        "torch.ao.quantization.quantizer.x86_inductor_quantizer",
        "X86InductorQuantizer",
        "get_default_x86_inductor_quantization_config",
    )]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            Quantizer  = getattr(m, cls)
            get_config = getattr(m, cfg)
            break
        except (ImportError, AttributeError) as exc:
            errors.append(f"  {mod_path}.{cls}: {exc}")

    if Quantizer is None:
        raise ImportError(
            "Cannot find X86InductorQuantizer.\n" + "\n".join(errors)
        )

    export_fn = None
    try:
        from torch.export import export as _export
        def export_fn(model, example_args):
            return _export(model, example_args).module()
    except (ImportError, AttributeError) as exc:
        errors.append(f"  torch.export.export: {exc}")

    if export_fn is None:
        try:
            from torch._export import capture_pre_autograd_graph as _cap
            def export_fn(model, example_args):
                return _cap(model, example_args)
        except (ImportError, AttributeError) as exc:
            errors.append(f"  torch._export.capture_pre_autograd_graph: {exc}")

    if export_fn is None:
        raise ImportError(
            "Cannot find graph export function.\n" + "\n".join(errors)
        )

    return prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn


def run_pt2e_ptq(fp32_model, test_loader,
                 flops_G: float, device) -> Optional[Dict]:
    """
    PT2E + X86InductorQuantizer + torch.compile.
    Fix 1: fresh calib loader used here too.
    size_mb = None — compiled graph cannot be trivially serialized.
    """
    print("\n        3b. PT2E + X86InductorQuantizer → torch.compile")
    os.makedirs(SAVE_DIR, exist_ok=True)

    try:
        (prepare_pt2e, convert_pt2e,
         Quantizer, get_config, export_fn) = _resolve_pt2e_imports()
        print(f"           PT2E imports resolved (torch {torch.__version__})")

        example_args = (torch.randn(1, 3, 32, 32, device=device),)
        m            = copy.deepcopy(fp32_model).to(device).eval()
        exported     = export_fn(m, example_args)

        quantizer = Quantizer()
        quantizer.set_global(get_config())
        prepared = prepare_pt2e(exported, quantizer)
        prepared.eval()

        # Fix 1: fresh calib loader
        fresh_calib = get_calib_loader()
        with torch.no_grad():
            for i, (imgs, _) in enumerate(fresh_calib):
                prepared(imgs.to(device))
                if i + 1 >= CALIB_BATCHES:
                    break

        q_pt2e = convert_pt2e(prepared)

        save_path = os.path.join(SAVE_DIR, "pt2e_int8.pth")
        try:
            torch.save(q_pt2e.state_dict(), save_path)
            print(f"        ✓ Saved (pre-compile state dict) → {save_path}")
        except Exception as exc:
            print(f"        ⚠ PT2E state-dict save skipped: {exc}")

        q_compiled = torch.compile(q_pt2e, mode="max-autotune", backend="inductor")

        metrics  = evaluate_torch(q_compiled, test_loader, device)
        infer_ms = measure_gpu_throughput(q_compiled, device)
        params   = count_params(q_pt2e, model_label="pt2e_inductor")

        result = build_entry(
            metrics      = metrics,
            params       = params,
            size_mb      = None,   # compiled graph not serializable
            flops_G      = flops_G,
            inference_ms = infer_ms,
            extra_notes  = {
                "note": (
                    "PT2E + torch.compile(inductor, max-autotune). "
                    "size_mb=null: compiled graph not trivially serializable. "
                    "Some ops may fall back to FP16 at compile time (PyTorch 2.3–2.5)."
                )
            },
        )
        print(
            f"           → Acc: {metrics['accuracy']:.4f}  "
            f"GPU: {infer_ms['batch128_gpu_ms']:.1f} ms"
        )
        return result

    except ImportError as exc:
        print(f"           → SKIPPED (PT2E unavailable): {exc}")
        return None
    except Exception as exc:
        print(f"           → FAILED: {exc}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# FP32 Baseline
# ══════════════════════════════════════════════════════════════════════════════
def benchmark_fp32(fp32_model, test_loader, flops_G: float, device) -> Dict:
    """Fix 11: uses canonical count_params (not bare sum)."""
    print("\n  [0/3] FP32 GPU Baseline")
    model    = copy.deepcopy(fp32_model).to(device).eval()
    metrics  = evaluate_torch(model, test_loader, device)
    infer_ms = measure_gpu_throughput(model, device)
    params   = count_params(model.cpu(), model_label="fp32_baseline")
    size_mb  = disk_size_mb(model.cpu())
    result   = build_entry(
        metrics      = metrics,
        params       = params,
        size_mb      = size_mb,
        flops_G      = flops_G,
        inference_ms = infer_ms,
    )
    print(
        f"        Acc: {metrics['accuracy']:.4f}  "
        f"GPU: {infer_ms['batch128_gpu_ms']:.1f} ms  "
        f"Disk: {size_mb:.2f} MB"
    )
    return result


# ══════════════════════════════════════════════════════════════════════════════
# Best Result Selection
# ══════════════════════════════════════════════════════════════════════════════
def select_best_result(output: Dict, baseline_acc: float) -> Dict:
    """
    Fix 8: select best quantized method and build _best_result section.
    Primary criterion : highest accuracy (lowest drop from baseline).
    Secondary criterion: smallest size_mb (for ties).
    pt2e_inductor excluded from size comparison if size_mb is None.
    """
    quant_entries = {
        k: v for k, v in output.items()
        if v is not None and k not in ("fp32_baseline", "_best_result", "_meta")
    }
    if not quant_entries:
        return {}

    best_key = min(
        quant_entries,
        key=lambda k: (
            baseline_acc - quant_entries[k]["accuracy"],
            quant_entries[k]["size_mb"] if quant_entries[k]["size_mb"] is not None else 999,
        ),
    )
    best      = quant_entries[best_key]
    fp32_lat  = output["fp32_baseline"]["inference_ms"]["batch128_gpu_ms"]
    best_lat  = best["inference_ms"]["batch128_gpu_ms"]
    speedup   = fp32_lat / best_lat if best_lat else 0.0
    acc_drop  = baseline_acc - best["accuracy"]

    return {
        "method"              : best_key,
        "accuracy"            : best["accuracy"],
        "f1"                  : best["f1"],
        "acc_drop"            : round(acc_drop, 6),
        "size_mb"             : best["size_mb"],
        "batch128_gpu_ms"     : best_lat,
        "throughput_imgs_sec" : best["inference_ms"]["throughput_imgs_sec"],
        "speedup_vs_fp32"     : round(speedup, 4),
        "flops_G"             : best["flops_G"],
        "params"              : best["params"],
        "params_nonzero"      : best["params_nonzero"],
        "active_provider"     : best.get("active_provider", "N/A"),
        "selection_criterion" : (
            "lowest accuracy drop vs FP32 baseline; "
            "size_mb as tiebreaker"
        ),
        "all_methods_accuracy": {
            k: v["accuracy"] for k, v in quant_entries.items()
        },
    }


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    parser = argparse.ArgumentParser(description="GPU PTQ — ResNet-50 / CIFAR-10")
    parser.add_argument("--install-deps", action="store_true")
    parser.add_argument("--device", default="cuda")
    args, _ = parser.parse_known_args()

    if args.install_deps:
        install_deps()
        return

    missing = []
    if torch is None: missing.append("torch          → pip install torch --index-url https://download.pytorch.org/whl/cu121")
    if onnx  is None: missing.append("onnx           → pip install onnx")
    if ort   is None: missing.append("onnxruntime-gpu → pip install onnxruntime-gpu")
    if missing:
        print("\n⚠  Missing required packages:")
        for m in missing:
            print(f"   {m}")
        sys.exit(1)

    device = torch.device(args.device if torch.cuda.is_available() else "cpu")

    SEP = "=" * 70
    print(f"\n{SEP}")
    print("  GPU PTQ — ResNet-50 / CIFAR-10")
    print(f"  Device : {device}")
    if torch.cuda.is_available():
        print(f"  GPU    : {torch.cuda.get_device_name(device)}")
        print(f"  CUDA   : {torch.version.cuda}")
        print(f"  cuDNN  : {torch.backends.cudnn.version()}")
    print("  Methods: ORT (MinMax|Entropy|Percentile) | TensorRT | Dynamic | PT2E")
    print(f"  Warmup : {WARMUP_RUNS} runs  |  Timing: {INFERENCE_RUNS} runs")
    print(SEP)

    if not os.path.exists(BASELINE_MODEL_PATH):
        print(f"\n✗ Baseline model not found: {BASELINE_MODEL_PATH}")
        sys.exit(1)

    fp32_model  = load_baseline(BASELINE_MODEL_PATH)
    test_loader = get_test_loader()

    # ── FLOPs — computed once on FP32, identical for all INT8 methods ─────
    print("\n  Computing FLOPs ...")
    flops_info = count_flops(fp32_model)
    flops_G    = flops_info["flops_G"]
    print(
        f"  FLOPs : {flops_G:.4f} GFLOPs  "
        f"Params: {flops_info['params_M']:.2f}M  "
        f"(tool: {flops_info['tool']})"
    )

    baseline_params = count_params(fp32_model, model_label="fp32_baseline")
    baseline_acc    = (
        json.load(open(BASELINE_METRICS_PATH))["accuracy"]
        if os.path.exists(BASELINE_METRICS_PATH)
        else None
    )

    output: Dict = {}

    # 0. FP32 baseline
    output["fp32_baseline"] = benchmark_fp32(fp32_model, test_loader, flops_G, device)
    if baseline_acc is None:
        baseline_acc = output["fp32_baseline"]["accuracy"]

    # 1. ORT — Fix 1: returns fp32_onnx path for TRT reuse
    print("\n  [1/3] ONNX Runtime PTQ")
    ort_results, fp32_onnx_path = run_onnx_ptq(
        fp32_model, test_loader, flops_G, baseline_params
    )
    for key, val in ort_results.items():
        if val is not None:
            output[key] = val

    # 2. TensorRT — reuse exported ONNX (avoid double export)
    print("\n  [2/3] TensorRT PTQ")
    trt_result = run_tensorrt_ptq(
        fp32_onnx_path, test_loader, flops_G, baseline_params
    )
    if trt_result is not None:
        output["tensorrt_int8"] = trt_result

    # Clean up shared FP32 ONNX
    if os.path.exists(fp32_onnx_path):
        os.remove(fp32_onnx_path)

    # 3a. Dynamic PTQ
    print("\n  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E")
    dyn_result = run_dynamic_ptq(fp32_model, test_loader, flops_G, device)
    if dyn_result is not None:
        output["dynamic_ptq"] = dyn_result

    # 3b. PT2E
    pt2e_result = run_pt2e_ptq(fp32_model, test_loader, flops_G, device)
    if pt2e_result is not None:
        output["pt2e_inductor"] = pt2e_result

    # ── Best result  (Fix 8) ──────────────────────────────────────────────
    output["_best_result"] = select_best_result(output, baseline_acc)

    # ── Meta  (Fix 12) ───────────────────────────────────────────────────
    output["_meta"] = {
        "script"       : "GPU PTQ — ResNet-50 / CIFAR-10",
        "device"       : str(device),
        "gpu"          : torch.cuda.get_device_name(device) if torch.cuda.is_available() else "CPU",
        "cuda_version" : torch.version.cuda if torch.cuda.is_available() else "N/A",
        "cudnn_version": str(torch.backends.cudnn.version()) if torch.cuda.is_available() else "N/A",
        "batch_size"   : BATCH_SIZE,
        "calib_images" : CALIB_SIZE,
        "calib_batches": CALIB_BATCHES,
        "inference_runs": INFERENCE_RUNS,
        "warmup_runs"  : WARMUP_RUNS,
        "flops_tool"   : flops_info["tool"],
        "flops_note"   : (
            "FLOPs identical for FP32 and INT8 — quantization changes "
            "dtype, not op count. GPU speedup from INT8 Tensor Cores."
        ),
        "timing_note"  : (
            "PyTorch/TRT: CUDA events (accurate GPU-only). "
            "ORT: wall-clock (time.perf_counter) — NOT directly comparable. "
            "ORT timings include Python/CUDA synchronization overhead."
        ),
        "size_note"    : (
            "PyTorch models: torch.save(model). "
            "ONNX: file size including QDQ graph metadata — can exceed FP32 "
            "torch.save size (format artifact, not real model size increase). "
            "TRT: serialized engine file."
        ),
        "calib_note"   : (
            "Fix applied: each ORT calibration method uses a fresh DataLoader. "
            "Original code shared one iterator — entropy/percentile received "
            "0 calibration batches, making all three methods effectively identical."
        ),
    }

    # ── Save JSON ──────────────────────────────────────────────────────────
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    # ── Console summary ────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print(f"  ✓ Results → {OUTPUT_JSON}")
    print(f"  ✓ Models  → {SAVE_DIR}/")
    print(f"\n  {'Method':<22} {'Acc':>7} {'Drop':>7} {'F1':>7} "
          f"{'Size MB':>8} {'Batch ms':>9} {'Imgs/s':>8} {'Provider'}")
    print(f"  {'-'*82}")

    for key, val in output.items():
        if key in ("_best_result", "_meta") or val is None:
            continue
        drop     = baseline_acc - val["accuracy"] if key != "fp32_baseline" else 0.0
        size_str = f"{val['size_mb']:.2f}" if val.get("size_mb") is not None else "   N/A"
        lat      = val["inference_ms"]["batch128_gpu_ms"]
        tput     = val["inference_ms"]["throughput_imgs_sec"]
        provider = val.get("active_provider", "CUDA events" if key != "fp32_baseline" else "torch")
        print(
            f"  {key:<22} {val['accuracy']:.4f}  {drop:>+.4f}  "
            f"{val['f1']:.4f}  {size_str:>8}  {lat:>9.1f}  "
            f"{tput:>8.1f}  {provider}"
        )

    if output.get("_best_result"):
        br = output["_best_result"]
        print(f"\n  ★ Best method  : {br['method']}")
        print(f"    Accuracy     : {br['accuracy']:.4f}  (drop: {br['acc_drop']:+.4f})")
        print(f"    Speedup      : {br['speedup_vs_fp32']:.2f}× vs FP32")
        size_str = f"{br['size_mb']:.2f} MB" if br['size_mb'] is not None else "N/A"
        print(f"    Size         : {size_str}")
        print(f"    Provider     : {br['active_provider']}")

    print(SEP + "\n")


if __name__ == "__main__":
    main()


  GPU PTQ — ResNet-50 / CIFAR-10
  Device : cuda
  GPU    : NVIDIA GeForce RTX 5070 Laptop GPU
  CUDA   : 12.8
  cuDNN  : 91900
  Methods: ORT (MinMax|Entropy|Percentile) | TensorRT | Dynamic | PT2E
  Warmup : 20 runs  |  Timing: 50 runs

  Computing FLOPs ...
  FLOPs : 2.5957 GFLOPs  Params: 23.52M  (tool: manual)
      count_params [fp32_baseline]: total=23,520,842  nonzero=23,520,842  sparsity=0.0000

  [0/3] FP32 GPU Baseline
      count_params [fp32_baseline]: total=23,520,842  nonzero=23,520,842  sparsity=0.0000
        Acc: 0.9320  GPU: 51.2 ms  Disk: 90.05 MB

  [1/3] ONNX Runtime PTQ

  [1/3] ONNX Runtime — Static INT8 PTQ (QDQ)


W0618 22:44:23.488000 68768 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0618 22:44:24.054000 68768 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
        ✓ ONNX exported → __1__saved_models_PTQ_GPU\resnet50_fp32_tmp.onnx  (0.22 MB, opset=17)
        ORT available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
        ORT requested providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

        [Minmax] calibrating ...


        [Minmax] active provider: CUDAExecutionProvider
        [Minmax] Acc: 0.9319  Disk: 90.18 MB  Batch: 64.1 ms  Provider: CUDAExecutionProvider
        ✓ Saved → __1__saved_models_PTQ_GPU\onnx_int8_minmax.onnx

        [Entropy] calibrating ...


Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 122
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128


        [Entropy] active provider: CUDAExecutionProvider
        [Entropy] Acc: 0.9319  Disk: 90.18 MB  Batch: 63.9 ms  Provider: CUDAExecutionProvider
        ✓ Saved → __1__saved_models_PTQ_GPU\onnx_int8_entropy.onnx

        [Percentile] calibrating ...


Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 122
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)


        [Percentile] active provider: CUDAExecutionProvider
        [Percentile] Acc: 0.9316  Disk: 90.18 MB  Batch: 63.8 ms  Provider: CUDAExecutionProvider
        ✓ Saved → __1__saved_models_PTQ_GPU\onnx_int8_percentile.onnx

  [2/3] TensorRT PTQ

  [2/3] TensorRT — INT8 PTQ
        ⚠ TensorRT/pycuda not available: No module named 'tensorrt'

  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E

        3a. Dynamic PTQ (Linear → qint8) on CUDA
        ✓ Saved → __1__saved_models_PTQ_GPU\dynamic_ptq_int8.pt
           → FAILED: Could not run 'quantized::linear_dynamic' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::linear_dynamic' is only available for these backends: [CPU, Meta, BackendSelect, Python, FuncTorchDynamicLayer